# Calculate auROC for first vs not-first lapses

Kendra Wyant  
May 19, 2026

### Set Up Environment

Packages, functions, and paths

In [ ]:

library(tidyverse)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

── Attaching packages ────────────────────────────────────── tidymodels 1.4.1 ──
✔ broom        1.0.12     ✔ rsample      1.3.2 
✔ dials        1.4.2      ✔ tailor       0.1.0 
✔ infer        1.1.0      ✔ tune         2.0.1 
✔ modeldata    1.5.1      ✔ workflows    1.3.0 
✔ parsnip      1.4.1      ✔ workflowsets 1.1.1 
✔ recipes      1.3.1      ✔ yardstick    1.3.2 
── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filter()   masks stats::filter()
✖ recipes::fixed()  masks stringr::fixed()
✖ dplyr::lag()      masks stats::lag()
✖ yardstick::spec() masks readr::spec()
✖ recipes::step()   masks stats::step()

ℹ SHA-1 hash of file is "0faa14c0c44c2635216370888b7da9bfa8d07979"

### Data

In [ ]:
preds_0 <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1day_0_v3_nested_strat_lh.rds"))

labels_0 <- read_csv(here::here(path_lag, "labels_1day_0lag.csv"),
                   show_col_types = FALSE)

preds_24 <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1day_24_v3_nested_strat_lh.rds"))

labels_24 <- read_csv(here::here(path_lag, "labels_1day_24lag.csv"),
                   show_col_types = FALSE)

preds_72 <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1day_72_v3_nested_strat_lh.rds"))

labels_72 <- read_csv(here::here(path_lag, "labels_1day_72lag.csv"),
                   show_col_types = FALSE)

preds_168 <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1day_168_v3_nested_strat_lh.rds"))

labels_168 <- read_csv(here::here(path_lag, "labels_1day_168lag.csv"),
                   show_col_types = FALSE)

preds_336 <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1day_336_v3_nested_strat_lh.rds"))

labels_336 <- read_csv(here::here(path_lag, "labels_1day_336lag.csv"),
                   show_col_types = FALSE)


### Create grouping variable

- first_lapse = Yes if no lapses on study or all data up until and including first lapse
- first_lapse = No for all data after first lapse

In [ ]:
# get first lapses
(first_lapses_0 <- labels_0 |> 
  filter(lapse == "yes") |> 
  group_by(subid) |> 
  arrange(dttm_label) |> 
  slice_head(n = 1) |> 
  ungroup())


# A tibble: 84 × 3
   subid dttm_label          lapse
   <dbl> <dttm>              <chr>
 1     2 2017-04-01 01:00:00 yes  
 2     3 2017-03-25 02:00:00 yes  
 3     7 2017-06-19 03:00:00 yes  
 4    10 2017-07-18 05:00:00 yes  
 5    11 2017-07-20 05:00:00 yes  
 6    16 2017-10-07 06:00:00 yes  
 7    19 2017-12-18 02:00:00 yes  
 8    20 2017-12-31 05:00:00 yes  
 9    25 2017-11-22 00:00:00 yes  
10    26 2017-11-16 03:00:00 yes  
# ℹ 74 more rows

 labels_0_full$first_lapse      n  percent
                        no 105007 0.382987
                       yes 169172 0.617013

# A tibble: 2 × 2
  first_lapse n_subid
  <chr>         <int>
1 no               82
2 yes             151

Combine with preds

In [ ]:
labels_0_full <- labels_0_full |>
  mutate(label = if_else(lapse == "no", "No lapse", "Lapse"),
         id_obs = row_number())  |> 
  select(-lapse) |> 
  right_join(preds_0, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))


### Calculate auROCs comparing up first lapse to no lapse and not first lapse to no lapse

First lapses

In [ ]:
auroc_first_lapse_0 <- labels_0_full |> 
  filter(first_lapse == "yes") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse_0 <- labels_0_full |> 
  filter(first_lapse == "no") |>
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs_0 <- auroc_first_lapse_0 |> 
  left_join(auroc_not_first_lapse_0, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.7933049, 0.8370734, 0.7383610, 0.6766002, 0.8588927,…
$ not_first_lapse <dbl> 0.8474285, 0.8577014, 0.8504305, 0.8914455, 0.7660186,…

In [ ]:
round(median(aurocs_0$first_lapse), 2)


[1] 0.79

[1] 0.84

[1] 0.9113233

### Get lagged aurocs

In [ ]:
first_lapses_24 <- labels_24 |> 
  filter(lapse == "yes") |> 
  group_by(subid) |> 
  arrange(dttm_label) |> 
  slice_head(n = 1) |> 
  ungroup()

first_lapses_72 <- labels_72 |> 
  filter(lapse == "yes") |> 
  group_by(subid) |> 
  arrange(dttm_label) |> 
  slice_head(n = 1) |> 
  ungroup()

first_lapses_168 <- labels_168 |> 
  filter(lapse == "yes") |> 
  group_by(subid) |> 
  arrange(dttm_label) |> 
  slice_head(n = 1) |> 
  ungroup()

first_lapses_336 <- labels_336 |> 
  filter(lapse == "yes") |> 
  group_by(subid) |> 
  arrange(dttm_label) |> 
  slice_head(n = 1) |> 
  ungroup()

labels_24_full <- labels_24 |> 
  mutate(first_lapse = if_else(!subid %in% first_lapses_24$subid, "yes", "no"))

labels_72_full <- labels_72 |> 
  mutate(first_lapse = if_else(!subid %in% first_lapses_72$subid, "yes", "no"))

labels_168_full <- labels_168 |> 
  mutate(first_lapse = if_else(!subid %in% first_lapses_168$subid, "yes", "no"))

labels_336_full <- labels_336 |> 
  mutate(first_lapse = if_else(!subid %in% first_lapses_336$subid, "yes", "no"))

for (the_subid in first_lapses_24$subid) {
  dttm <- first_lapses_24 |> 
    filter(subid == the_subid) |> 
    pull(dttm_label)
 
  # filter out full 24 hours of lapse
  
  row_ids <- which(
    labels_24_full$subid == the_subid &
    (labels_24_full$dttm_label <= dttm |
       (labels_24_full$dttm_label > dttm & labels_24_full$dttm_label < dttm + hours(24)))
  )

  labels_24_full[row_ids, "first_lapse"] <- "yes"
}

for (the_subid in first_lapses_72$subid) {
  dttm <- first_lapses_72 |> 
    filter(subid == the_subid) |> 
    pull(dttm_label)
 
  # filter out full 24 hours of lapse
  
  row_ids <- which(
    labels_72_full$subid == the_subid &
    (labels_72_full$dttm_label <= dttm |
       (labels_72_full$dttm_label > dttm & labels_72_full$dttm_label < dttm + hours(24)))
  )

  labels_72_full[row_ids, "first_lapse"] <- "yes"
}

for (the_subid in first_lapses_168$subid) {
  dttm <- first_lapses_168 |> 
    filter(subid == the_subid) |> 
    pull(dttm_label)
 
  # filter out full 24 hours of lapse
  
  row_ids <- which(
    labels_168_full$subid == the_subid &
    (labels_168_full$dttm_label <= dttm |
       (labels_168_full$dttm_label > dttm & labels_168_full$dttm_label < dttm + hours(24)))
  )

  labels_168_full[row_ids, "first_lapse"] <- "yes"
}

for (the_subid in first_lapses_336$subid) {
  dttm <- first_lapses_336 |> 
    filter(subid == the_subid) |> 
    pull(dttm_label)
 
  # filter out full 24 hours of lapse
  
  row_ids <- which(
    labels_336_full$subid == the_subid &
    (labels_336_full$dttm_label <= dttm |
       (labels_336_full$dttm_label > dttm & labels_336_full$dttm_label < dttm + hours(24)))
  )

  labels_336_full[row_ids, "first_lapse"] <- "yes"
}


Combine with preds

In [ ]:
labels_24_full <- labels_24_full |>
  mutate(label = if_else(lapse == "no", "No lapse", "Lapse"),
         id_obs = row_number())  |> 
  select(-lapse) |> 
  right_join(preds_24, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))

labels_72_full <- labels_72_full |>
  mutate(label = if_else(lapse == "no", "No lapse", "Lapse"),
         id_obs = row_number())  |> 
  select(-lapse) |> 
  right_join(preds_72, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))

labels_168_full <- labels_168_full |>
  mutate(label = if_else(lapse == "no", "No lapse", "Lapse"),
         id_obs = row_number())  |> 
  select(-lapse) |> 
  right_join(preds_168, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))

labels_336_full <- labels_336_full |>
  mutate(label = if_else(lapse == "no", "No lapse", "Lapse"),
         id_obs = row_number())  |> 
  select(-lapse) |> 
  right_join(preds_336, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))


Calculate aurocs

In [ ]:
auroc_first_lapse_24 <- labels_24_full |> 
  filter(first_lapse == "yes") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse_24 <- labels_24_full |> 
  filter(first_lapse == "no") |>
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs_24 <- auroc_first_lapse_24 |> 
  left_join(auroc_not_first_lapse_24, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.7795215, 0.7861322, 0.7173364, 0.6717013, 0.8566131,…
$ not_first_lapse <dbl> 0.7844837, 0.8008565, 0.8180832, 0.8303189, 0.7432217,…

[1] 0.76

[1] 0.8

[1] 0.8898142

In [ ]:
auroc_first_lapse_72 <- labels_72_full |> 
  filter(first_lapse == "yes") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse_72 <- labels_72_full |> 
  filter(first_lapse == "no") |>
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs_72 <- auroc_first_lapse_72 |> 
  left_join(auroc_not_first_lapse_72, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.7533207, 0.8336034, 0.6931045, 0.6858374, 0.9024679,…
$ not_first_lapse <dbl> 0.7303143, 0.7775176, 0.7974761, 0.8048731, 0.7320877,…

[1] 0.76

[1] 0.77

[1] 0.8811099

In [ ]:
auroc_first_lapse_168 <- labels_168_full |> 
  filter(first_lapse == "yes") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse_168 <- labels_168_full |> 
  filter(first_lapse == "no") |>
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs_168 <- auroc_first_lapse_168 |> 
  left_join(auroc_not_first_lapse_168, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.7692811, 0.8343768, 0.7083643, 0.7118573, 0.9035027,…
$ not_first_lapse <dbl> 0.6989571, 0.7405818, 0.7698609, 0.8091646, 0.6907782,…

[1] 0.77

[1] 0.74

[1] 0.8683958

In [ ]:
auroc_first_lapse_336 <- labels_336_full |> 
  filter(first_lapse == "yes") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse_336 <- labels_336_full |> 
  filter(first_lapse == "no") |>
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs_336 <- auroc_first_lapse_336 |> 
  left_join(auroc_not_first_lapse_336, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.6678881, 0.7767964, 0.7209289, 0.7865845, 0.8868538,…
$ not_first_lapse <dbl> 0.6170462, 0.6479898, 0.7687413, 0.7522558, 0.6284988,…

[1] 0.79

[1] 0.69

[1] 0.8471884